# ShiftGuard-X — NB92: Causal Spatiotemporal ULP-Lifting Forest (CSULF)

**Status: experimental mechanism screening, not an established novel algorithm.** This notebook tests a new *ShiftGuard-X candidate* — not a claim that its components have never been published. It uses a calibration-frozen, acyclic cross-station dependency graph and an exactly reversible modular difference of ordered IEEE-754 bit patterns.

**Why this experiment?** NB89 verified 32 Beijing streams but the temporal XOR scheme was worse than native PCodec on the 16 primary PM2.5/PM10 streams. This candidate jointly encodes time-aligned sensors; NB89 did **not** test joint decoding. However, graph predictors (Corra/C3/Virtual), temporal/spatial difference transforms, and ordered-float residuals (ULPack 2026) are prior art. Any paper would need a real mechanism distinction **and** statistically replicated gains.

**Input:** official UCI Beijing Multi-Site Air Quality dataset 501, downloaded directly with original ZIP SHA-256 logged. Optional upload of `SGX_NB89_results.zip` for held-out digest parity / PCodec size reference. **Never replace a download error with synthetic research data.** Synthetic data are used in an *explicitly labeled local correctness test only*.

**Protocol:** 8 station streams × PM2.5, PM10 (primary), SO2 and NO2 (controls), 32,000 values each, sorted chronologically, 25% calibration and 75% held out. Every compressed package is fully self-describing, includes dependency metadata, and reconstructs every original float64 bit, including special NaN payloads. No post-test codec/mode selection.

**Predetermined feasibility criterion:** in **both** PM2.5 and PM10, at least 5% aggregate bytes saved against the smallest comparable stronger baseline among native PCodec sum, separate Zstandard/LZMA, and joint Zstandard/LZMA; pass all exactness/source gates. This is only a local feasibility test: one Beijing family is insufficient for publication-level generalization. Explicit failures => REJECT, not 'tune until positive'.

In [ ]:
# Colab CPU setup. A failure is recorded later as UNAVAILABLE, not treated as a win.
import sys, subprocess, importlib.util
SETUP={}
for module, package in [('zstandard','zstandard==0.25.0'),('pcodec','pcodec==1.0.3')]:
    if importlib.util.find_spec(module) is None:
        try:
            r=subprocess.run([sys.executable,'-m','pip','-q','install',package],capture_output=True,text=True,timeout=180)
            SETUP[module]={'returncode':r.returncode,'stderr':r.stderr[-300:]}
        except Exception as e: SETUP[module]={'error':repr(e)}
    else: SETUP[module]='already installed'
print(SETUP)

In [ ]:
import os, io, sys, json, csv, time, struct, lzma, zlib, hashlib, zipfile, platform, urllib.request, statistics
from pathlib import Path
import numpy as np, pandas as pd
try: import zstandard as zstd
except ImportError: zstd=None
try: from pcodec import standalone as pcs
except ImportError: pcs=None
try: import pcodec
except ImportError: pcodec=None
ROOT=Path('/content') if Path('/content').exists() else Path('/mnt/data')
OUT=ROOT/'SGX_NB92_outputs';OUT.mkdir(parents=True,exist_ok=True)
STATIONS=['Aotizhongxin','Changping','Dongsi','Guanyuan','Gucheng','Shunyi','Wanliu','Wanshouxigong']
CHANNELS=['PM2.5','PM10','SO2','NO2']
URL='https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip'
N=32000;CUT=N//4
U=np.uint64;SIGN=U(1<<63);ALL=U(0xffffffffffffffff)
SHA=lambda b:hashlib.sha256(b).hexdigest()
RAW=lambda a:np.asarray(a,dtype='<f8').copy().tobytes()
def writecsv(filename,rows):
    if not rows:rows=[{'status':'NO_ROWS'}]
    fields=list(dict.fromkeys(k for r in rows for k in r))
    with (OUT/filename).open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows)
def writejson(filename,d):(OUT/filename).write_text(json.dumps(d,indent=2,default=str))

def ordered(bits):
    bits=np.asarray(bits,dtype='<u8')
    return np.where((bits&SIGN)!=0,~bits,bits^SIGN).astype('<u8',copy=False)
def unorder(k):
    k=np.asarray(k,dtype='<u8')
    return np.where((k&SIGN)!=0,k^SIGN,~k).astype('<u8',copy=False)
def zz(r):
    r=np.asarray(r,dtype='<u8')
    return ((r<<U(1)) ^ (U(0)-(r>>U(63)))).astype('<u8')
def unzz(r):
    r=np.asarray(r,dtype='<u8')
    return ((r>>U(1)) ^ (U(0)-(r&U(1)))).astype('<u8')
def diff(x):
    o=x.copy();o[1:]=x[1:]-x[:-1];return o
def undiff(d):return np.cumsum(d,dtype=np.uint64)
def transform(child,parent,mode):
    if mode=='self':return diff(child)
    if parent is None:raise ValueError('cross mode requires parent')
    if mode=='cross':return child-parent
    if mode=='double':return diff(child-parent)
    raise ValueError(mode)
def untransform(r,parent,mode):
    if mode=='self':return undiff(r)
    if parent is None:raise ValueError('cross mode requires parent')
    if mode=='cross':return r+parent
    if mode=='double':return undiff(r)+parent
    raise ValueError(mode)
def shuf(b):return np.frombuffer(b,dtype=np.uint8).reshape(-1,8).T.copy().tobytes()
def unshuf(b,n):return np.frombuffer(b,dtype=np.uint8).reshape(8,n).T.copy().tobytes()
def zipped(b,shuffle=False,codec='zstd3'):
    if shuffle:b=shuf(b)
    if codec=='zstd3':
        if zstd is None:raise RuntimeError('zstandard unavailable')
        return zstd.ZstdCompressor(level=3).compress(b)
    if codec=='lzma1':return lzma.compress(b,preset=1)
    raise ValueError(codec)
def unzipped(b,n,shuffle=False,codec='zstd3'):
    if codec=='zstd3':d=zstd.ZstdDecompressor().decompress(b,max_output_size=n*8)
    elif codec=='lzma1':d=lzma.decompress(b)
    else:raise ValueError(codec)
    if len(d)!=n*8:raise ValueError('Decoded length mismatch')
    return unshuf(d,n) if shuffle else d
print('Runtime:',sys.version.split()[0],'Zstandard available',zstd is not None,'PCodec available',pcs is not None)

### Correctness: arbitrary IEEE-754 bits and reversible 2D transform

Ordered-key mapping is a permutation over all 64-bit words, **including distinct NaN payloads**. Arithmetic operates modulo 2^64; no floating-point calculation is used in reconstruction. The `double` mode is a separable causal lifting step: encode a temporal difference of the child–parent ordered-key difference. Transform components are established coding ideas; the proposed frozen graph and its measured interaction are the experimental question.

In [ ]:
rng=np.random.default_rng(92001)
bit=np.concatenate([np.array([0,1,2,0x8000000000000000,0x7ff0000000000000,0xfff0000000000000,0x7ff8000000000001,0x7ff8000000000002,0xffffffffffffffff],dtype='<u8'),rng.integers(0,2**64,size=501,dtype=np.uint64)])
pbits=rng.integers(0,2**64,size=len(bit),dtype=np.uint64)
child=ordered(bit);par=ordered(pbits)
assert np.array_equal(unorder(child),bit)
assert np.array_equal(unzz(zz(bit)),bit)
edge=[]
for mode in ['self','cross','double']:
    r=transform(child,par if mode!='self' else None,mode)
    result=untransform(unzz(zz(r)),par if mode!='self' else None,mode)
    assert np.array_equal(result,child),mode
    edge.append({'mode':mode,'status':'PASS_BITWISE','tested_words':len(bit)})
writecsv('edge_checks.csv',edge)
print(edge)

### Fully self-describing joint file format and bit-exact decoder

Frame = 6-byte signature `SGX92\0` + 4-byte JSON-header length + UTF-8 canonical JSON (channel, timestamp row count, ordered station names, predictor metadata, coding methods, payload byte lengths) + concatenated compressed station payloads. Raw source ZIP is **not** part of the decoder. Each station is reconstructed in order, so parent values always exist. Malformed or truncated frames fail closed.

In [ ]:
MAGIC=b'SGX92\0'
def encode_frame(rawmatrix,channel,policies,station_order):
    if zstd is None:raise RuntimeError('zstandard dependency missing')
    # Raw bytes MUST be retained: interpreting float values numerically can destroy a NaN payload.
    bits=np.asarray(rawmatrix,dtype='<u8')
    if bits.shape[1]!=len(station_order):raise ValueError('station matrix shape mismatch')
    n=bits.shape[0];keys=ordered(bits)
    compressed=[];desc=[]
    for pos,idx in enumerate(station_order):
        pol=policies[idx];parentidx=pol['parent']
        if parentidx is not None and parentidx not in station_order[:pos]:raise ValueError('noncausal station parent')
        residual=transform(keys[:,idx],keys[:,parentidx] if parentidx is not None else None,pol['mode'])
        packed=zipped(zz(residual).astype('<u8').tobytes(),pol['shuffle'])
        compressed.append(packed)
        desc.append({'station':STATIONS[idx],'station_index':idx,'parent':parentidx,'mode':pol['mode'],'shuffle':pol['shuffle'],'payload_len':len(packed)})
    h={'format':'SGX92-CAUSAL-LIFT-V1','channel':channel,'n':n,'stations':len(station_order),'codec':'zstd3','descriptors':desc}
    header=json.dumps(h,separators=(',',':'),sort_keys=True).encode()
    return MAGIC+struct.pack('<I',len(header))+header+b''.join(compressed)
def decode_frame(blob):
    if len(blob)<10 or blob[:6]!=MAGIC:raise ValueError('Invalid SGX92 magic')
    ln,=struct.unpack('<I',blob[6:10])
    if not 0<ln<100000 or 10+ln>len(blob):raise ValueError('Bad JSON header length')
    h=json.loads(blob[10:10+ln]);ds=h['descriptors'];n=h['n']
    if n<1 or n>10000000 or not ds or len(ds)!=h['stations']:raise ValueError('Bad frame shape')
    keys={};start=10+ln
    for desc in ds:
        idx=desc['station_index'];pidx=desc['parent'];size=desc['payload_len']
        if idx in keys or (pidx is not None and pidx not in keys) or size<1 or start+size>len(blob):raise ValueError('Bad dependency / length')
        p=blob[start:start+size];start+=size
        r=unzz(np.frombuffer(unzipped(p,n,desc['shuffle']),dtype='<u8'))
        keys[idx]=untransform(r,keys[pidx] if pidx is not None else None,desc['mode'])
    if start!=len(blob):raise ValueError('Unexpected trailing bytes')
    return h,np.stack([unorder(keys[i]) for i in sorted(keys)],axis=1)
# Whole-frame control over arbitrary bitwords / two sensor channels
syn=np.column_stack([bit,pbits]).astype('<u8')
p=[{'parent':None,'mode':'self','shuffle':True},{'parent':0,'mode':'double','shuffle':True}]
spec=encode_frame(syn,'EDGE',p,[0,1]); hh, got=decode_frame(spec)
assert np.array_equal(got,syn)
for bad in [spec[:-1],spec[:8],spec[:5]+b'X'+spec[6:]]:
    try:decode_frame(bad);raise AssertionError('corrupt frame erroneously accepted')
    except (ValueError,zlib.error,lzma.LZMAError):pass
print('Whole-frame random-bit round-trip and truncated-frame rejection: PASS')

### Download source, preserve exact NB89 parsing

To prevent a mismatch, this cell reproduces the earlier 32,000-row, four-channel, chronologically sorted parser and records the original archive checksum. The timestamp-aligned matrix is built from the 8 selected station datasets. No interpolation, data repair, or synthetic substitution is permitted.

In [ ]:
def csvfiles(archive,depth=0,prefix=''):
    if depth>3:raise ValueError('ZIP recursion depth exceeded')
    with zipfile.ZipFile(io.BytesIO(archive)) as z:
        for name in z.namelist():
            if name.lower().endswith('.zip'):
                yield from csvfiles(z.read(name),depth+1,prefix+name+'::')
            elif name.lower().endswith('.csv'):
                yield prefix+name,z.read(name)
source_ok=False;DATA={};source_errors=[];provenance=[]
try:
    req=urllib.request.Request(URL,headers={'User-Agent':'ShiftGuard-X-NB92'})
    with urllib.request.urlopen(req,timeout=100) as f:archive=f.read()
    found=list(csvfiles(archive))
    source_sha=SHA(archive)
    provenance=[{'source':'UCI-501','url':URL,'sha256':source_sha,'bytes':len(archive),'csv_members':len(found)}]
    assert source_sha=='b04da438b2f331ac0ffd45aebdfec0d20d2367feb5f6948c4b1f7ce1191e33c4','Source version does not match NB89'
    by_station={}
    for station in STATIONS:
        matches=[(n,b) for n,b in found if 'PRSA_Data_'+station+'_' in n]
        if len(matches)!=1:raise ValueError(f'{station}: CSV members {len(matches)}')
        d=pd.read_csv(io.BytesIO(matches[0][1])).sort_values(['year','month','day','hour'],kind='stable')
        if len(d)<N:raise ValueError('Too few station rows')
        by_station[station]=d
    for channel in CHANNELS:
        vecs=[]
        for station in STATIONS:
            vals=pd.to_numeric(by_station[station][channel],errors='coerce').to_numpy(dtype='<f8')[:N]
            vecs.append(vals.view('<u8').copy())
        DATA[channel]=np.stack(vecs,axis=1)
    source_ok=True
except Exception as e:source_errors=[{'stage':'source','error':repr(e)}]
writecsv('source_provenance.csv',provenance)
writecsv('source_errors.csv',source_errors)
print('REAL UCI DATA READY:',source_ok,'channels:',list(DATA),'errors:',source_errors)

### Import optional NB89 reference archive and validate same held-out bit streams

This is optional for general-purpose benchmarking; **without matched verified native PCodec bytes, the study must not declare a win over NB89 specialized methods**. If NB89 archive is not at `/content/SGX_NB89_results.zip` you can upload it with the Colab file sidebar and rerun this cell.

In [ ]:
old=ROOT/'SGX_NB89_results.zip';REF={}
if old.exists():
    try:
        with zipfile.ZipFile(old) as z:
            for r in csv.DictReader(io.StringIO(z.read('heldout_frozen_decisions.csv').decode())):
                REF[(r['station'],r['channel'])]=r
        print('NB89 matched reference loaded:',len(REF))
    except Exception as e:print('NB89 reference incompatible:',repr(e));REF={}
else:print('NB89 reference archive not provided; native comparison must be obtained by PCodec rerun in this notebook.')

### Calibration-only graph policy selection

For each channel the graph grows greedily. Each new station chooses the cheapest *calibration-only* policy from self-temporal, aligned-parent spatial, and second-order spatiotemporal modular residual modes. Every mode may use byte splitting. This is an algorithm-selection trial and its cost must be reported; policies are fixed before the held-out values are encoded.

In [ ]:
def select_forest(train_bits):
    n,cols=train_bits.shape;keys=ordered(train_bits);avail=list(range(cols));order=[];policies=[None]*cols; trials=0
    t0=time.perf_counter()
    while avail:
        best=None
        for ix in avail:
            for mode,parent in [('self',None)]+[(m,p) for p in order for m in ('cross','double')]:
                residual=transform(keys[:,ix],keys[:,parent] if parent is not None else None,mode)
                payload=zz(residual).astype('<u8').tobytes()
                for shuffle in (False,True):
                    enc=zipped(payload,shuffle);trials+=1
                    choice=(len(enc),ix,mode,parent,shuffle)
                    if best is None or choice<best:best=choice
        _,ix,mode,parent,shuffle=best
        order.append(ix);policies[ix]={'mode':mode,'parent':parent,'shuffle':shuffle}
        avail.remove(ix)
    return order,policies,trials,(time.perf_counter()-t0)*1000

def independent_baselines(test_bits):
    vals=[np.ascontiguousarray(test_bits[:,i],dtype='<u8').tobytes() for i in range(test_bits.shape[1])]
    raw_column=np.ascontiguousarray(test_bits.T).tobytes()
    raw_row=np.ascontiguousarray(test_bits).tobytes()
    summary={}
    joint_meta=22+len(json.dumps(STATIONS))
    for m in ('zstd3','lzma1'):
        summary['separate_'+m]=sum(len(zipped(v,codec=m)) + 22 for v in vals)
        summary['separate_shuffle_'+m]=sum(len(zipped(v,shuffle=True,codec=m)) + 22 for v in vals)
        for ordering,b in [('col',raw_column),('row',raw_row)]:
            summary['joint_'+ordering+'_'+m]=len(zipped(b,codec=m))+joint_meta
            summary['joint_'+ordering+'_shuffle_'+m]=len(zipped(b,shuffle=True,codec=m))+joint_meta
    if pcs is not None:
        try:
            summary['native_pcodec_sum']=sum(len(bytes(pcs.simple_compress(np.frombuffer(v,dtype='<f8'),pcodec.ChunkConfig())))+22 for v in vals)
        except Exception as e:summary['pcodec_error']=repr(e)
    return summary
print('Methods defined.')

### Run the blind held-out comparison

It is **not permitted** to select a codec or change the graph after seeing these sizes. Reference samples must have identical SHA-256 values and lengths before results are merged. The `raw` baseline includes no additional data transformations; other baseline input bytes are identical.

In [ ]:
rows=[];mode_rows=[];failures=[];outputs={};BASE_BY_CHANNEL={};POL_BY_CHANNEL={};ORDER_BY_CHANNEL={}
if source_ok and zstd is not None:
    for channel in CHANNELS:
        full=DATA[channel];train=full[:CUT];hold=full[CUT:]
        order,pol,trials,cal_ms=select_forest(train)
        POL_BY_CHANNEL[channel]=pol;ORDER_BY_CHANNEL[channel]=order
        t0=time.perf_counter_ns()
        frame=encode_frame(hold,channel,pol,order)
        encode_ms=(time.perf_counter_ns()-t0)/1e6
        t0=time.perf_counter_ns()
        meta,restored=decode_frame(frame)
        decode_ms=(time.perf_counter_ns()-t0)/1e6
        if not np.array_equal(restored,hold):raise AssertionError('Held-out IEEE bit mismatch for '+channel)
        base=independent_baselines(hold)
        # NB89 comparison only when held-out input hashes confirm identical arrays
        matched=True;oldsize=0
        for i,st in enumerate(STATIONS):
            v=hold[:,i].astype('<u8').tobytes();digest=SHA(v)
            r=REF.get((st,channel))
            if r is None or r.get('heldout_sha256')!=digest or int(float(r.get('n_test',0)))!=len(hold):matched=False
            else:oldsize+=int(r['native_bytes'])
        if matched:base['NB89_pcodec_size_matched']=oldsize
        generic=[v for k,v in base.items() if (k.startswith('separate_') or k.startswith('joint_') or k.startswith('native_') or k=='NB89_pcodec_size_matched') and isinstance(v,int)]
        strongest=min(generic) if generic else None
        winning=(strongest is not None and len(frame)<strongest)
        record={'channel':channel,'primary':channel in ('PM2.5','PM10'),'stations':len(STATIONS),'heldout_values':hold.size,'heldout_sha256':SHA(np.ascontiguousarray(hold).tobytes()),'candidate_bytes':len(frame),'strongest_baseline_bytes':strongest,'savings_vs_strongest_pct':100*(strongest-len(frame))/strongest if strongest else None,'calibration_trials':trials,'calibration_ms':round(cal_ms,3),'encode_ms':round(encode_ms,3),'decode_ms':round(decode_ms,3),'exact_bitwise':True,'all_NB89_hashes_matched':matched,**base}
        rows.append(record);BASE_BY_CHANNEL[channel]=base
        for idx in order:mode_rows.append({'channel':channel,'station':STATIONS[idx],**pol[idx],'graph_rank':order.index(idx)})
        outputs[channel]=frame
        print(channel,'candidate',len(frame),'strongest',strongest,'gain %',round(record['savings_vs_strongest_pct'],2),'NB89 hash matched',matched)
else:failures.append({'stage':'benchmark','error':'UCI source download or zstandard unavailable'})
writecsv('channel_results.csv',rows)
writecsv('frozen_graph_policies.csv',mode_rows)
writecsv('experiment_errors.csv',failures)

### Controlled ablations and synchronization negative control

**Self-only ablation:** select independently (calibration-only) whether each station uses ordinary ordered-key temporal differences with or without byte split, and encode exactly the same held-out values in the exact same joint SGX92 container. This tests whether the cross-station graph contributes beyond a univariate transform.

**Station dephasing test:** independently circular-shift the *held-out* streams by seeded large offsets, leaving every station's local time-series distribution and adjacency largely unchanged but disrupting across-station alignment. Do not retrain the graph; the **original frozen policies** are applied. This is a diagnostic negative control, not an additional dataset or evidence of algorithmic originality.

In [ ]:
ablation=[];phase_rng=np.random.default_rng(92002)
if source_ok and zstd is not None and len(rows)==4:
    for channel in CHANNELS:
        train=DATA[channel][:CUT];test=DATA[channel][CUT:]
        order=ORDER_BY_CHANNEL[channel];pol=POL_BY_CHANNEL[channel]
        temp_keys=ordered(train)
        selfpol=[]
        for i in range(len(STATIONS)):
            r=zz(diff(temp_keys[:,i])).astype('<u8').tobytes()
            v=min((len(zipped(r,q)),q) for q in (False,True))
            selfpol.append({'mode':'self','parent':None,'shuffle':v[1]})
        f_self=encode_frame(test,channel,selfpol,list(range(len(STATIONS))))
        assert np.array_equal(decode_frame(f_self)[1],test)
        shift_min=min(300,max(1,len(test)//10))
        shifts=phase_rng.integers(shift_min,len(test)-shift_min,size=len(STATIONS))
        shifted=np.column_stack([np.roll(test[:,i],int(shifts[i])) for i in range(len(STATIONS))]).astype('<u8')
        f_shifted=encode_frame(shifted,channel,pol,order)
        assert np.array_equal(decode_frame(f_shifted)[1],shifted)
        ablation.append({'channel':channel,'forest_original_bytes':len(outputs[channel]),'self_only_bytes':len(f_self),
                         'forest_vs_self_gain_pct':100*(len(f_self)-len(outputs[channel]))/len(f_self),
                         'forest_shifted_bytes':len(f_shifted),'original_to_dephased_increase_pct':100*(len(f_shifted)-len(outputs[channel]))/len(outputs[channel]),
                         'dephasing_seed':92002,'shift_offsets_json':json.dumps(shifts.tolist()),'bitwise_roundtrip':'PASS'})
writecsv('mechanism_ablations.csv',ablation)
print('Ablation results:',ablation)

### Decision gate and archive export

Acceptance here means **only** the experiment merits a genuine independent-families replication and originality audit. Even if savings exceed 5%, the candidate is not automatically a new research contribution: ordered integer/ULP deltas, reversible spatiotemporal lifting, graphs, entropy coding and adaptation are all known individually. Compare explicit methodology against Corra, C3, Virtual, ULPack (Oct 2026), ACTSD (2026) and structure-aware battery sensing (Oct 2026).

In [ ]:
eligible=len(rows)==4 and not failures and all(r['exact_bitwise'] for r in rows)
primary=[r for r in rows if r['primary']]
strong_ok=all(r.get('native_pcodec_sum') or r.get('NB89_pcodec_size_matched') for r in primary)
gains=all(r['savings_vs_strongest_pct']>=5 for r in primary) if len(primary)==2 else False
mechanism_win=all(a['forest_vs_self_gain_pct']>0 for a in ablation if a['channel'] in ('PM2.5','PM10')) and len([a for a in ablation if a['channel'] in ('PM2.5','PM10')])==2
if not eligible:state='INCOMPLETE'
elif not strong_ok:state='INCOMPLETE_NATIVE_BASELINE'
elif gains and mechanism_win:state='FEASIBLE_REPLICATION_AND_NOVELTY_AUDIT_REQUIRED'
else:state='REJECT_THIS_CANDIDATE_ON_BEIJING'
reason={
 'candidate_name':'Causal Spatiotemporal ULP-Lifting Forest (CSULF)',
 'status':state, 'source_verified':source_ok, 'channels_complete':len(rows),
 'primary_results':[{k:v for k,v in r.items() if k in ('channel','candidate_bytes','strongest_baseline_bytes','savings_vs_strongest_pct','all_NB89_hashes_matched')} for r in primary],
 'bitwise_tests_passed':len(edge)==3,'cross_station_ablation_win_on_both_primary_channels':mechanism_win,'research_novelty_established':False,
 'one_dataset_family_only':True,'Q1_submittable_new_algorithm':False,
 'next_action':'Run genuinely independent dataset families, benchmark native comparable ALP/Chimp/Corra and audit methodological novelty' if state.startswith('FEASIBLE') else 'Stop this candidate; do not post-hoc tune on Beijing if it fails',
 'warnings':['Native PCodec must be present (or matched original NB89 archive) for a strong baseline','All policies frozen by 25% calibration only','Comparable native ALP/Corra not benchmarked by this pilot','Any accepted mechanism must be demonstrably different from relevant prior work']
}
writejson('decision.json',reason)
writejson('environment.json',{'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'pandas':pd.__version__,'zstandard':getattr(zstd,'__version__','UNAVAILABLE'),'installation':SETUP})
(OUT/'research_scope.txt').write_text('CSULF is a candidate built from known components. Pilot covers one Beijing dataset family. No novelty percentage or publication-ready claim is established.\n')
archive=ROOT/'SGX_NB92_results.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file():z.write(p,p.name)
print(json.dumps(reason,indent=2))
print('DOWNLOAD:',archive)
try:
    from google.colab import files;files.download(str(archive))
except ImportError:pass